# Resume ArcFace + ConvNeXt-Tiny Training on Colab
Continue training from the Google Drive checkpoint `arcface_convnext_epoch20.pth` without starting from scratch.
This notebook restores the model and optimizer state, resumes at epoch 21, trains to epoch 70, uses `CosineAnnealingLR`, keeps batch size 32, adds light `RandomRotation(15)` augmentation, and saves refreshed checkpoints back to Drive.

In [ ]:
# 1) Install the only extra dependency used by the notebook
!pip install -q timm==0.9.2

In [ ]:
# 2) Mount Google Drive and define the paths used for resume training
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/kaplumbaga_tanima'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'datasets')
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
CHECKPOINT_IN = os.path.join(PROJECT_ROOT, 'checkpoints', 'arcface_convnext_epoch20.pth')
CHECKPOINT_OUT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', 'resume_arcface')
os.makedirs(CHECKPOINT_OUT_DIR, exist_ok=True)

print('TRAIN_DIR:', TRAIN_DIR)
print('CHECKPOINT_IN:', CHECKPOINT_IN)
print('CHECKPOINT_OUT_DIR:', CHECKPOINT_OUT_DIR)

In [ ]:
# 3) Imports, seed, and training configuration
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import timm

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

BATCH_SIZE = 32
IMG_SIZE = 224
INITIAL_RESUME_EPOCH = 20
START_EPOCH = INITIAL_RESUME_EPOCH + 1
TOTAL_EPOCHS = 70
RESUME_EPOCHS = TOTAL_EPOCHS - START_EPOCH + 1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Resume phase epochs:', RESUME_EPOCHS)

In [ ]:
# 4) Dataset and dataloader with light augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(f'Train directory not found: {TRAIN_DIR}')

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
NUM_CLASSES = len(train_dataset.classes)
print('Classes:', NUM_CLASSES)
print('Samples:', len(train_dataset))
print('First 5 classes:', train_dataset.classes[:5])

In [ ]:
# 5) ArcFace + ConvNeXt-Tiny model definition
class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False, ls_eps=0.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.ls_eps = ls_eps
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_tensor, label):
        cosine = F.linear(F.normalize(input_tensor), F.normalize(self.weight))
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=0.0))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        if self.ls_eps > 0:
            one_hot = (1 - self.ls_eps) * one_hot + self.ls_eps / self.out_features
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

class ArcFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, backbone_name='convnext_tiny.fb_in22k_ft_in1k', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        backbone_dim = self.backbone.num_features
        self.bn1 = nn.BatchNorm1d(backbone_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(backbone_dim, embedding_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.metric_fc = ArcFace(embedding_dim, num_classes, s=30.0, m=0.50, ls_eps=0.1)

        nn.init.constant_(self.bn1.weight, 1)
        nn.init.constant_(self.bn1.bias, 0)
        nn.init.kaiming_normal_(self.fc.weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.bn2.weight, 1)
        nn.init.constant_(self.bn2.bias, 0)

    def forward(self, x, labels=None):
        x = self.backbone(x)
        x = self.bn1(x)
        x = self.dropout(x)
        embeddings = self.fc(x)
        embeddings = self.bn2(embeddings)
        if labels is None:
            return embeddings
        logits = self.metric_fc(embeddings, labels)
        return logits, embeddings

In [ ]:
# 6) Model, optimizer, scheduler, and resume state loading
model = ArcFaceModel(NUM_CLASSES, embedding_dim=512).to(DEVICE)
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': [p for p in model.bn1.parameters()] + [p for p in model.fc.parameters()] + [p for p in model.bn2.parameters()], 'lr': 1e-4},
    {'params': model.metric_fc.parameters(), 'lr': 1e-4},
], weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RESUME_EPOCHS, eta_min=1e-6)

if not os.path.isfile(CHECKPOINT_IN):
    raise FileNotFoundError(f'Resume checkpoint not found: {CHECKPOINT_IN}')

checkpoint = torch.load(CHECKPOINT_IN, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
if 'metric_fc_state_dict' not in checkpoint:
    raise KeyError('Checkpoint is missing metric_fc_state_dict; resume cannot continue safely.')
model.metric_fc.load_state_dict(checkpoint['metric_fc_state_dict'])
if 'optimizer_state_dict' not in checkpoint:
    raise KeyError('Checkpoint is missing optimizer_state_dict; resume cannot continue safely.')
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

checkpoint_epoch = checkpoint.get('epoch', INITIAL_RESUME_EPOCH)
best_acc = float(checkpoint.get('acc', 0.0))
if checkpoint_epoch not in (INITIAL_RESUME_EPOCH, INITIAL_RESUME_EPOCH - 1):
    print(f'Warning: checkpoint epoch metadata is {checkpoint_epoch}, but this notebook is configured to resume from epoch 21.')

print('Loaded checkpoint:', CHECKPOINT_IN)
print('Starting from epoch:', START_EPOCH)
print('Best accuracy carried forward:', best_acc)

In [ ]:
# 7) Resume training loop and save checkpoints back to Drive
def save_checkpoint(epoch, best_acc_value, tag='latest'):
    payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'metric_fc_state_dict': model.metric_fc.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'acc': best_acc_value,
        'batch_size': BATCH_SIZE,
        'image_size': IMG_SIZE,
        'num_classes': NUM_CLASSES,
    }
    epoch_path = os.path.join(CHECKPOINT_OUT_DIR, f'arcface_convnext_epoch{epoch}.pth')
    latest_path = os.path.join(CHECKPOINT_OUT_DIR, 'arcface_convnext_latest.pth')
    tag_path = os.path.join(CHECKPOINT_OUT_DIR, f'arcface_convnext_{tag}.pth')
    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    torch.save(payload, tag_path)
    return epoch_path, latest_path, tag_path

for epoch in range(START_EPOCH, TOTAL_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, total=len(train_loader), leave=False)
    for images, labels in loop:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits, embeddings = model(images, labels)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()

        loop.set_description(f'Epoch [{epoch}/{TOTAL_EPOCHS}]')
        loop.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100.0 * correct / max(total, 1):.2f}%')

    scheduler.step()
    epoch_loss = running_loss / max(total, 1)
    epoch_acc = 100.0 * correct / max(total, 1)
    previous_best_acc = best_acc
    is_best = epoch_acc >= previous_best_acc
    best_acc = max(best_acc, epoch_acc)

    epoch_path, latest_path, tag_path = save_checkpoint(epoch, best_acc, tag='best' if is_best else 'latest')
    print(f'Epoch {epoch}/{TOTAL_EPOCHS} | loss={epoch_loss:.4f} | acc={epoch_acc:.2f}%')
    print(f'Saved: {epoch_path}')

final_path = os.path.join(CHECKPOINT_OUT_DIR, 'arcface_convnext_epoch70_final.pth')
final_payload = {
    'epoch': TOTAL_EPOCHS,
    'model_state_dict': model.state_dict(),
    'metric_fc_state_dict': model.metric_fc.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'acc': best_acc,
    'batch_size': BATCH_SIZE,
    'image_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
}
torch.save(final_payload, final_path)
print('Final checkpoint:', final_path)

### Resume Training Notes
- Keep the class folder order unchanged between the checkpoint and the current Drive dataset, or `metric_fc` loading will break.
- If the checkpoint was saved without `optimizer_state_dict`, exact resume training is not possible.
- If the runtime resets, remount Drive and rerun the path / checkpoint load cells before continuing.
- The notebook saves `epoch` checkpoints, a `latest` checkpoint, a `best` checkpoint, and a final epoch 70 checkpoint back to Drive.